# Step 5 — Effect of Sensor Degradation on RUL Prediction

Objectives:

- reload the frozen FD001 preprocessing artifacts
- reload the selected clean GRU without retraining
- reproduce the clean validation baseline
- apply the validated Step-4 sensor corruptions
- quantify degradation in RUL prediction accuracy
- identify vulnerable sensors and corruption types
- analyze severity-response behavior
- examine dangerous RUL overestimation near end-of-life

Important rules:

- official NASA test data remain untouched
- the clean GRU remains frozen
- preprocessing remains frozen
- corruption is applied only to validation copies
- Step-4 corruption definitions are not changed based on model results

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

# Locate project root
cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root."
    )


RAW_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "raw"
    / "CMAPSSData"
)

INTERIM_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "interim"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "processed"
    / "FD001"
)

DEGRADATION_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "sensor_degradation"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "models"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "06_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root       :", PROJECT_ROOT)
print("Raw exists         :", RAW_DIR.exists())
print("Interim exists     :", INTERIM_DIR.exists())
print("Processed exists   :", PROCESSED_DIR.exists())
print("Degradation exists :", DEGRADATION_DIR.exists())
print("Models exists      :", MODELS_DIR.exists())

In [ ]:
with open(
    PROCESSED_DIR
    / "fd001_preprocessing_config.json",
    "r"
) as f:
    config = json.load(f)


with open(
    DEGRADATION_DIR
    / "fd001_corruption_specification.json",
    "r"
) as f:
    corruption_spec = json.load(f)


scaler = joblib.load(
    PROCESSED_DIR
    / "fd001_standard_scaler.joblib"
)


split_df = pd.read_csv(
    INTERIM_DIR
    / "fd001_engine_split_seed42.csv"
)


sensor_reference_stats = pd.read_csv(
    DEGRADATION_DIR
    / "fd001_sensor_reference_stats.csv",
    index_col=0
)


PRIMARY_INPUT_COLUMNS = config[
    "primary_input_columns"
]

RETAINED_SENSOR_COLUMNS = [
    col
    for col in PRIMARY_INPUT_COLUMNS
    if col.startswith("sensor_")
]

WINDOW_SIZE = 30
TARGET_COLUMN = "RUL_cap_125"
SEED = 42


print("Primary inputs   :", len(PRIMARY_INPUT_COLUMNS))
print("Retained sensors :", len(RETAINED_SENSOR_COLUMNS))
print("Window size      :", WINDOW_SIZE)
print("Target           :", TARGET_COLUMN)
print("Seed             :", SEED)

In [ ]:
GRU_MODEL_PATH = (
    MODELS_DIR
    / "fd001_gru_selected_cap125_window30_seed42.pt"
)

print(
    "Selected GRU exists:",
    GRU_MODEL_PATH.exists()
)

print(
    "Model path:",
    GRU_MODEL_PATH
)

In [ ]:
column_names = (
    ["unit_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

fd001_raw = pd.read_csv(
    RAW_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

val_engine_ids = set(
    split_df.loc[
        split_df["split"] == "validation",
        "unit_id"
    ]
)

val_raw = (
    fd001_raw[
        fd001_raw["unit_id"].isin(
            val_engine_ids
        )
    ]
    .sort_values(
        ["unit_id", "cycle"]
    )
    .reset_index(drop=True)
)


def add_rul_targets(df):
    df = df.copy()

    max_cycle = (
        df.groupby("unit_id")["cycle"]
        .transform("max")
    )

    df["RUL_raw"] = (
        max_cycle - df["cycle"]
    )

    df["RUL_cap_125"] = np.minimum(
        df["RUL_raw"],
        125
    )

    df["RUL_cap_130"] = np.minimum(
        df["RUL_raw"],
        130
    )

    return df


val_raw = add_rul_targets(
    val_raw
)

print("Validation rows   :", len(val_raw))
print(
    "Validation engines:",
    val_raw["unit_id"].nunique()
)

In [ ]:
val_clean_scaled = val_raw.copy()

val_clean_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_raw[
        PRIMARY_INPUT_COLUMNS
    ]
)

print(
    "Missing scaled values:",
    val_clean_scaled[
        PRIMARY_INPUT_COLUMNS
    ].isna().sum().sum()
)

print(
    "Infinite scaled values:",
    np.isinf(
        val_clean_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    ).sum()
)

In [ ]:
def create_sequences(
    df,
    feature_columns,
    target_column,
    window_size
):
    X = []
    y = []
    metadata = []

    for unit_id, group in df.groupby(
        "unit_id",
        sort=True
    ):

        group = (
            group
            .sort_values("cycle")
            .reset_index(drop=True)
        )

        features = group[
            feature_columns
        ].to_numpy(
            dtype=np.float32
        )

        targets = group[
            target_column
        ].to_numpy(
            dtype=np.float32
        )

        for end_idx in range(
            window_size - 1,
            len(group)
        ):

            start_idx = (
                end_idx
                - window_size
                + 1
            )

            X.append(
                features[
                    start_idx:
                    end_idx + 1
                ]
            )

            y.append(
                targets[end_idx]
            )

            metadata.append({
                "unit_id":
                    int(unit_id),

                "cycle":
                    int(
                        group.loc[
                            end_idx,
                            "cycle"
                        ]
                    ),

                "RUL_raw":
                    float(
                        group.loc[
                            end_idx,
                            "RUL_raw"
                        ]
                    ),

                "RUL_cap_125":
                    float(
                        group.loc[
                            end_idx,
                            "RUL_cap_125"
                        ]
                    )
            })

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            y,
            dtype=np.float32
        ),
        pd.DataFrame(metadata)
    )


X_val_clean, y_val_clean, val_seq_meta = (
    create_sequences(
        df=val_clean_scaled,
        feature_columns=PRIMARY_INPUT_COLUMNS,
        target_column=TARGET_COLUMN,
        window_size=WINDOW_SIZE
    )
)

print(
    "Validation sequence shape:",
    X_val_clean.shape
)

print(
    "Validation target shape  :",
    y_val_clean.shape
)

In [ ]:
class GRURegressor(nn.Module):

    def __init__(
        self,
        input_size=17,
        hidden_size=64
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.regressor = nn.Sequential(
            nn.Linear(
                hidden_size,
                32
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        gru_output, _ = self.gru(x)

        last_hidden = (
            gru_output[:, -1, :]
        )

        prediction = self.regressor(
            last_hidden
        )

        return prediction.squeeze(-1)

In [ ]:
checkpoint = torch.load(
    GRU_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

if isinstance(checkpoint, dict):
    print(
        "Checkpoint keys:",
        list(checkpoint.keys())
    )

if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):
    state_dict = checkpoint[
        "model_state_dict"
    ]

elif (
    isinstance(checkpoint, dict)
    and "state_dict" in checkpoint
):
    state_dict = checkpoint[
        "state_dict"
    ]

elif (
    isinstance(checkpoint, dict)
    and "model_state" in checkpoint
):
    state_dict = checkpoint[
        "model_state"
    ]

elif (
    isinstance(checkpoint, dict)
    and all(
        torch.is_tensor(v)
        for v in checkpoint.values()
    )
):
    # Checkpoint itself is a state_dict
    state_dict = checkpoint

else:
    raise ValueError(
        "Could not identify model state "
        "inside the checkpoint."
    )


clean_gru = GRURegressor(
    input_size=len(
        PRIMARY_INPUT_COLUMNS
    ),
    hidden_size=64
)

clean_gru.load_state_dict(
    state_dict
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

clean_gru = clean_gru.to(
    DEVICE
)

clean_gru.eval()

print("Device:", DEVICE)
print("Frozen GRU loaded successfully.")

In [ ]:
print("Checkpoint best epoch :", checkpoint["best_epoch"])
print("Checkpoint best RMSE  :", checkpoint["best_val_rmse"])
print("Checkpoint window     :", checkpoint["window_size"])
print("Checkpoint target     :", checkpoint["target"])
print("Checkpoint seed       :", checkpoint["seed"])

assert checkpoint["window_size"] == WINDOW_SIZE
assert checkpoint["target"] == TARGET_COLUMN
assert checkpoint["seed"] == SEED

assert list(
    checkpoint["input_columns"]
) == list(
    PRIMARY_INPUT_COLUMNS
)

print(
    "Checkpoint metadata matches the frozen Step-5 configuration."
)

In [ ]:
def predict_gru(
    model,
    X,
    device,
    batch_size=512
):
    model.eval()

    predictions = []

    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = (
                start + batch_size
            )

            batch = torch.tensor(
                X[start:end],
                dtype=torch.float32,
                device=device
            )

            batch_predictions = (
                model(batch)
                .detach()
                .cpu()
                .numpy()
                .ravel()
            )

            predictions.append(
                batch_predictions
            )

    return np.concatenate(
        predictions
    )


clean_predictions = predict_gru(
    model=clean_gru,
    X=X_val_clean,
    device=DEVICE
)

print(
    "Clean predictions shape:",
    clean_predictions.shape
)

print(
    "Prediction minimum:",
    clean_predictions.min()
)

print(
    "Prediction maximum:",
    clean_predictions.max()
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)


def regression_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    rmse = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


clean_full_metrics = (
    regression_metrics(
        y_val_clean,
        clean_predictions
    )
)

print(
    "Clean full-validation metrics:"
)

for metric, value in (
    clean_full_metrics.items()
):
    print(
        f"{metric}: {value:.3f}"
    )

In [ ]:
assert np.isclose(
    clean_full_metrics["RMSE"],
    checkpoint["best_val_rmse"],
    rtol=1e-5,
    atol=1e-4
)

print(
    "Clean full-validation RMSE reproduces the saved checkpoint."
)

In [ ]:
EVAL_FRACTIONS = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

snapshot_records = []

for unit_id, group in val_raw.groupby(
    "unit_id",
    sort=True
):

    max_cycle = int(
        group["cycle"].max()
    )

    for fraction in EVAL_FRACTIONS:

        snapshot_cycle = int(
            max_cycle * fraction
        )

        snapshot_records.append({
            "unit_id":
                int(unit_id),

            "cycle":
                snapshot_cycle,

            "eval_fraction":
                fraction
        })


snapshot_lookup = pd.DataFrame(
    snapshot_records
)

print(
    "Snapshot rows:",
    len(snapshot_lookup)
)

print(
    "Snapshot engines:",
    snapshot_lookup[
        "unit_id"
    ].nunique()
)

display(
    snapshot_lookup.head(10)
)

In [ ]:
clean_prediction_df = (
    val_seq_meta.copy()
)

clean_prediction_df[
    "target"
] = y_val_clean

clean_prediction_df[
    "prediction"
] = clean_predictions

In [ ]:
clean_snapshot_df = (
    snapshot_lookup.merge(
        clean_prediction_df,
        on=[
            "unit_id",
            "cycle"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "Balanced snapshot rows:",
    len(clean_snapshot_df)
)

print(
    "Missing predictions:",
    clean_snapshot_df[
        "prediction"
    ].isna().sum()
)

assert len(
    clean_snapshot_df
) == 100

assert (
    clean_snapshot_df[
        "prediction"
    ].isna().sum()
    == 0
)

display(
    clean_snapshot_df.head(10)
)

In [ ]:
clean_snapshot_metrics = (
    regression_metrics(
        clean_snapshot_df[
            "target"
        ].to_numpy(),

        clean_snapshot_df[
            "prediction"
        ].to_numpy()
    )
)

print(
    "Clean balanced-snapshot metrics:"
)

for metric, value in (
    clean_snapshot_metrics.items()
):
    print(
        f"{metric}: {value:.3f}"
    )

In [ ]:
assert np.isclose(
    clean_snapshot_metrics["RMSE"],
    14.380,
    atol=0.01
)

assert np.isclose(
    clean_snapshot_metrics["MAE"],
    10.382,
    atol=0.01
)

assert np.isclose(
    clean_snapshot_metrics["R2"],
    0.789,
    atol=0.01
)

print(
    "Step-3 clean balanced-snapshot baseline reproduced successfully."
)

In [ ]:
CORRUPTION_LEVELS = corruption_spec["levels"]

CORRUPTION_SEED = corruption_spec[
    "corruption_seed"
]

print("Corruption seed:", CORRUPTION_SEED)
print("Corruption levels:")

for corruption, levels in (
    CORRUPTION_LEVELS.items()
):
    print(
        corruption,
        ":",
        levels
    )

In [ ]:
def add_sensor_noise(
    df,
    sensor,
    severity,
    sensor_stats,
    seed=42
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    noise_std = (
        severity
        * sensor_std
    )

    rng = np.random.default_rng(
        seed
    )

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=len(corrupted)
    )

    corrupted[sensor] = (
        corrupted[sensor].to_numpy()
        + noise
    )

    return corrupted, noise


print(
    "Step-4 additive-noise function ready."
)

In [ ]:
TEST_CORRUPTION = "noise"
TEST_SENSOR = "sensor_2"
TEST_SEVERITY = "medium"

TEST_ALPHA = CORRUPTION_LEVELS[
    TEST_CORRUPTION
][TEST_SEVERITY]


val_corrupted_raw, injected_noise = (
    add_sensor_noise(
        df=val_raw,
        sensor=TEST_SENSOR,
        severity=TEST_ALPHA,
        sensor_stats=sensor_reference_stats,
        seed=CORRUPTION_SEED
    )
)


print("Corruption :", TEST_CORRUPTION)
print("Sensor     :", TEST_SENSOR)
print("Severity   :", TEST_SEVERITY)
print("Alpha      :", TEST_ALPHA)

print(
    "Noise mean :",
    injected_noise.mean()
)

print(
    "Noise std  :",
    injected_noise.std(ddof=0)
)

In [ ]:
val_corrupted_scaled = (
    val_corrupted_raw.copy()
)

val_corrupted_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_corrupted_raw[
        PRIMARY_INPUT_COLUMNS
    ]
)


print(
    "Missing scaled values:",
    val_corrupted_scaled[
        PRIMARY_INPUT_COLUMNS
    ].isna().sum().sum()
)

print(
    "Infinite scaled values:",
    np.isinf(
        val_corrupted_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    ).sum()
)

In [ ]:
(
    X_val_corrupted,
    y_val_corrupted,
    corrupted_seq_meta
) = create_sequences(
    df=val_corrupted_scaled,
    feature_columns=PRIMARY_INPUT_COLUMNS,
    target_column=TARGET_COLUMN,
    window_size=WINDOW_SIZE
)


print(
    "Corrupted sequence shape:",
    X_val_corrupted.shape
)

print(
    "Corrupted target shape:",
    y_val_corrupted.shape
)

In [ ]:
assert np.array_equal(
    y_val_corrupted,
    y_val_clean
)

assert np.array_equal(
    corrupted_seq_meta[
        ["unit_id", "cycle"]
    ].to_numpy(),
    val_seq_meta[
        ["unit_id", "cycle"]
    ].to_numpy()
)

print(
    "Clean and corrupted evaluation samples are perfectly aligned."
)

In [ ]:
corrupted_predictions = predict_gru(
    model=clean_gru,
    X=X_val_corrupted,
    device=DEVICE
)

print(
    "Corrupted predictions shape:",
    corrupted_predictions.shape
)

assert (
    corrupted_predictions.shape
    ==
    clean_predictions.shape
)

print(
    "Frozen-GRU corrupted predictions generated successfully."
)

In [ ]:
corrupted_full_metrics = (
    regression_metrics(
        y_val_corrupted,
        corrupted_predictions
    )
)

print("Clean full-validation:")
for metric, value in (
    clean_full_metrics.items()
):
    print(
        f"{metric}: {value:.3f}"
    )


print("\nCorrupted full-validation:")
for metric, value in (
    corrupted_full_metrics.items()
):
    print(
        f"{metric}: {value:.3f}"
    )


rmse_change = (
    corrupted_full_metrics["RMSE"]
    - clean_full_metrics["RMSE"]
)

rmse_change_pct = (
    100
    * rmse_change
    / clean_full_metrics["RMSE"]
)

print(
    "\nRMSE change:",
    round(rmse_change, 3)
)

print(
    "RMSE change (%):",
    round(rmse_change_pct, 2)
)

In [ ]:
def add_sensor_bias(
    df,
    sensor,
    severity,
    direction,
    sensor_stats
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be either -1 or +1."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    bias_amount = (
        direction
        * severity
        * sensor_std
    )

    corrupted[sensor] = (
        corrupted[sensor]
        + bias_amount
    )

    return corrupted, bias_amount


def add_sensor_drift(
    df,
    sensor,
    severity,
    direction,
    sensor_stats
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be either -1 or +1."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    final_drift = (
        direction
        * severity
        * sensor_std
    )

    drift_values = pd.Series(
        0.0,
        index=corrupted.index,
        dtype=float
    )

    for unit_id, group in corrupted.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values(
            "cycle"
        )

        progress = np.linspace(
            0.0,
            1.0,
            len(group)
        )

        drift_values.loc[
            group.index
        ] = (
            final_drift
            * progress
        )

    corrupted[sensor] = (
        corrupted[sensor]
        + drift_values
    )

    return (
        corrupted,
        drift_values.to_numpy(),
        final_drift
    )


def add_sensor_dropout(
    df,
    sensor,
    dropout_rate,
    sensor_stats,
    seed=42
):
    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if not 0.0 <= dropout_rate < 1.0:
        raise ValueError(
            "dropout_rate must be between 0 and 1."
        )

    missing_df = df.copy()

    n_rows = len(
        missing_df
    )

    n_drop = int(
        round(
            dropout_rate
            * n_rows
        )
    )

    rng = np.random.default_rng(
        seed
    )

    random_order = rng.permutation(
        n_rows
    )

    drop_positions = random_order[
        :n_drop
    ]

    drop_mask = np.zeros(
        n_rows,
        dtype=bool
    )

    drop_mask[
        drop_positions
    ] = True

    drop_index = missing_df.index[
        drop_positions
    ]

    missing_df.loc[
        drop_index,
        sensor
    ] = np.nan

    impute_value = sensor_stats.loc[
        sensor,
        "mean"
    ]

    imputed_df = missing_df.copy()

    imputed_df[sensor] = (
        imputed_df[sensor]
        .fillna(impute_value)
    )

    return (
        missing_df,
        imputed_df,
        drop_mask,
        impute_value
    )


def add_stuck_sensor(
    df,
    sensor,
    affected_fraction
):
    if sensor not in RETAINED_SENSOR_COLUMNS:
        raise ValueError(
            f"{sensor} is not a retained sensor."
        )

    if not 0.0 < affected_fraction < 1.0:
        raise ValueError(
            "affected_fraction must be between 0 and 1."
        )

    corrupted = df.copy()

    affected_mask = pd.Series(
        False,
        index=corrupted.index
    )

    stuck_records = []

    for unit_id, group in corrupted.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values(
            "cycle"
        )

        n_rows = len(group)

        n_affected = int(
            round(
                affected_fraction
                * n_rows
            )
        )

        n_affected = max(
            1,
            n_affected
        )

        n_affected = min(
            n_affected,
            n_rows - 1
        )

        onset_position = (
            n_rows
            - n_affected
        )

        indices = (
            group.index
            .to_numpy()
        )

        source_index = indices[
            onset_position - 1
        ]

        affected_indices = indices[
            onset_position:
        ]

        freeze_value = (
            corrupted.loc[
                source_index,
                sensor
            ]
        )

        corrupted.loc[
            affected_indices,
            sensor
        ] = freeze_value

        affected_mask.loc[
            affected_indices
        ] = True

        stuck_records.append({
            "unit_id":
                unit_id,

            "trajectory_length":
                n_rows,

            "requested_fraction":
                affected_fraction,

            "affected_count":
                n_affected,

            "actual_fraction":
                n_affected / n_rows,

            "source_cycle":
                group.iloc[
                    onset_position - 1
                ]["cycle"],

            "stuck_start_cycle":
                group.iloc[
                    onset_position
                ]["cycle"],

            "freeze_value":
                freeze_value
        })

    stuck_info = pd.DataFrame(
        stuck_records
    )

    return (
        corrupted,
        affected_mask.to_numpy(),
        stuck_info
    )


print(
    "All five frozen Step-4 corruption functions are ready."
)

In [ ]:
def apply_corruption(
    clean_df,
    corruption,
    sensor,
    severity_name,
    direction=None,
    seed=42
):
    if corruption not in CORRUPTION_LEVELS:
        raise ValueError(
            f"Unknown corruption: {corruption}"
        )

    if sensor not in RETAINED_SENSOR_COLUMNS:
        raise ValueError(
            f"Unknown retained sensor: {sensor}"
        )

    level = CORRUPTION_LEVELS[
        corruption
    ][severity_name]

    if corruption == "noise":

        corrupted_df, _ = add_sensor_noise(
            df=clean_df,
            sensor=sensor,
            severity=level,
            sensor_stats=sensor_reference_stats,
            seed=seed
        )

    elif corruption == "bias":

        if direction not in (-1, 1):
            raise ValueError(
                "Bias requires direction -1 or +1."
            )

        corrupted_df, _ = add_sensor_bias(
            df=clean_df,
            sensor=sensor,
            severity=level,
            direction=direction,
            sensor_stats=sensor_reference_stats
        )

    elif corruption == "drift":

        if direction not in (-1, 1):
            raise ValueError(
                "Drift requires direction -1 or +1."
            )

        (
            corrupted_df,
            _,
            _
        ) = add_sensor_drift(
            df=clean_df,
            sensor=sensor,
            severity=level,
            direction=direction,
            sensor_stats=sensor_reference_stats
        )

    elif corruption == "dropout":

        (
            _,
            corrupted_df,
            _,
            _
        ) = add_sensor_dropout(
            df=clean_df,
            sensor=sensor,
            dropout_rate=level,
            sensor_stats=sensor_reference_stats,
            seed=seed
        )

    elif corruption == "stuck":

        (
            corrupted_df,
            _,
            _
        ) = add_stuck_sensor(
            df=clean_df,
            sensor=sensor,
            affected_fraction=level
        )

    else:
        raise ValueError(
            f"Unsupported corruption: {corruption}"
        )

    return corrupted_df

In [ ]:
print(
    "Unified corruption dispatcher ready."
)

In [ ]:
def evaluate_corruption_scenario(
    corruption,
    sensor,
    severity_name,
    direction=None,
    seed=42
):
    # --------------------------------
    # 1. Apply frozen corruption
    # --------------------------------
    corrupted_raw = apply_corruption(
        clean_df=val_raw,
        corruption=corruption,
        sensor=sensor,
        severity_name=severity_name,
        direction=direction,
        seed=seed
    )

    # --------------------------------
    # 2. Frozen scaling
    # --------------------------------
    corrupted_scaled = (
        corrupted_raw.copy()
    )

    corrupted_scaled[
        PRIMARY_INPUT_COLUMNS
    ] = scaler.transform(
        corrupted_raw[
            PRIMARY_INPUT_COLUMNS
        ]
    )

    X_corrupt, y_corrupt, meta = (
        create_sequences(
            df=corrupted_scaled,
            feature_columns=
                PRIMARY_INPUT_COLUMNS,
            target_column=
                TARGET_COLUMN,
            window_size=
                WINDOW_SIZE
        )
    )

    # --------------------------------
    # 3. Strict alignment checks
    # --------------------------------
    assert np.array_equal(
        y_corrupt,
        y_val_clean
    )

    assert np.array_equal(
        meta[
            ["unit_id", "cycle"]
        ].to_numpy(),
        val_seq_meta[
            ["unit_id", "cycle"]
        ].to_numpy()
    )

    # --------------------------------
    # 4. Frozen GRU prediction
    # --------------------------------
    predictions = predict_gru(
        model=clean_gru,
        X=X_corrupt,
        device=DEVICE
    )

    # --------------------------------
    # 5. Full-validation metrics
    # --------------------------------
    full_metrics = regression_metrics(
        y_corrupt,
        predictions
    )

    # --------------------------------
    # 6. Balanced snapshots
    # --------------------------------
    prediction_df = (
        meta[
            ["unit_id", "cycle"]
        ]
        .copy()
    )

    prediction_df[
        "target"
    ] = y_corrupt

    prediction_df[
        "prediction"
    ] = predictions

    snapshot_df = (
        snapshot_lookup.merge(
            prediction_df,
            on=[
                "unit_id",
                "cycle"
            ],
            how="left",
            validate="one_to_one"
        )
    )

    assert (
        snapshot_df[
            "prediction"
        ]
        .isna()
        .sum()
        == 0
    )

    snapshot_metrics = (
        regression_metrics(
            snapshot_df[
                "target"
            ].to_numpy(),

            snapshot_df[
                "prediction"
            ].to_numpy()
        )
    )

    # --------------------------------
    # 7. Store comparison
    # --------------------------------
    result = {
        "corruption":
            corruption,

        "sensor":
            sensor,

        "severity":
            severity_name,

        "level":
            CORRUPTION_LEVELS[
                corruption
            ][severity_name],

        "direction":
            direction,

        "seed":
            seed,

        "full_RMSE":
            full_metrics["RMSE"],

        "full_MAE":
            full_metrics["MAE"],

        "full_R2":
            full_metrics["R2"],

        "full_RMSE_delta":
            full_metrics["RMSE"]
            - clean_full_metrics["RMSE"],

        "full_RMSE_change_pct":
            100
            * (
                full_metrics["RMSE"]
                - clean_full_metrics["RMSE"]
            )
            / clean_full_metrics["RMSE"],

        "snapshot_RMSE":
            snapshot_metrics["RMSE"],

        "snapshot_MAE":
            snapshot_metrics["MAE"],

        "snapshot_R2":
            snapshot_metrics["R2"],

        "snapshot_RMSE_delta":
            snapshot_metrics["RMSE"]
            - clean_snapshot_metrics["RMSE"],

        "snapshot_RMSE_change_pct":
            100
            * (
                snapshot_metrics["RMSE"]
                - clean_snapshot_metrics["RMSE"]
            )
            / clean_snapshot_metrics["RMSE"]
    }

    return (
        result,
        predictions,
        snapshot_df
    )

In [ ]:
(
    test_result,
    test_predictions,
    test_snapshot_df
) = evaluate_corruption_scenario(
    corruption="noise",
    sensor="sensor_2",
    severity_name="medium",
    direction=None,
    seed=CORRUPTION_SEED
)

display(
    pd.DataFrame(
        [test_result]
    ).round(4)
)

In [ ]:
assert np.isclose(
    test_result["full_RMSE"],
    corrupted_full_metrics["RMSE"],
    rtol=1e-8,
    atol=1e-8
)

assert np.allclose(
    test_predictions,
    corrupted_predictions,
    rtol=1e-6,
    atol=1e-6
)

print(
    "Reusable degradation-impact pipeline "
    "reproduces the manual test exactly."
)

In [ ]:
scenario_records = []

scenario_id = 1

for corruption in [
    "noise",
    "bias",
    "drift",
    "dropout",
    "stuck"
]:

    for sensor in RETAINED_SENSOR_COLUMNS:

        for severity_name in [
            "low",
            "medium",
            "high"
        ]:

            # Bias and drift have two directions
            if corruption in [
                "bias",
                "drift"
            ]:

                directions = [-1, 1]

            else:

                directions = [0]

            for direction in directions:

                scenario_records.append({
                    "scenario_id":
                        f"FD001_S{scenario_id:03d}",

                    "corruption":
                        corruption,

                    "sensor":
                        sensor,

                    "severity":
                        severity_name,

                    "level":
                        CORRUPTION_LEVELS[
                            corruption
                        ][severity_name],

                    "direction":
                        direction
                })

                scenario_id += 1


scenario_plan_df = pd.DataFrame(
    scenario_records
)

display(
    scenario_plan_df.head(15)
)

print(
    "Total planned scenarios:",
    len(scenario_plan_df)
)

print("\nScenarios by corruption:")

display(
    scenario_plan_df
    .groupby("corruption")
    .size()
    .rename("scenarios")
    .to_frame()
)

In [ ]:
assert len(
    scenario_plan_df
) == 315

assert (
    scenario_plan_df[
        "scenario_id"
    ].is_unique
)

print(
    "Complete 315-scenario experiment plan verified."
)

In [ ]:
STEP5_DIR = (
    RESULTS_DIR
    / "step5_degradation_impact"
)

STEP5_DIR.mkdir(
    parents=True,
    exist_ok=True
)

scenario_plan_path = (
    STEP5_DIR
    / "fd001_degradation_scenario_plan.csv"
)

scenario_plan_df.to_csv(
    scenario_plan_path,
    index=False
)

print(
    "Scenario plan saved to:"
)

print(
    scenario_plan_path
)

In [ ]:
SCREENING_PATH = (
    STEP5_DIR
    / "fd001_degradation_impact_screening_seed42.csv"
)


# ---------------------------------------
# Resume previous run if one exists
# ---------------------------------------
if SCREENING_PATH.exists():

    screening_results_df = pd.read_csv(
        SCREENING_PATH
    )

    completed_ids = set(
        screening_results_df[
            "scenario_id"
        ]
    )

    results_records = (
        screening_results_df
        .to_dict("records")
    )

    print(
        "Existing results found:",
        len(completed_ids)
    )

else:

    completed_ids = set()
    results_records = []

    print(
        "Starting a new screening run."
    )


# ---------------------------------------
# Run remaining scenarios
# ---------------------------------------
total_scenarios = len(
    scenario_plan_df
)

for row_number, row in (
    scenario_plan_df.iterrows()
):

    scenario_id = row[
        "scenario_id"
    ]

    if scenario_id in completed_ids:
        continue

    corruption = row[
        "corruption"
    ]

    sensor = row[
        "sensor"
    ]

    severity = row[
        "severity"
    ]

    direction_value = int(
        row["direction"]
    )

    # Non-directional corruptions
    if direction_value == 0:
        direction_arg = None
    else:
        direction_arg = direction_value


    (
        result,
        _,
        _
    ) = evaluate_corruption_scenario(
        corruption=corruption,
        sensor=sensor,
        severity_name=severity,
        direction=direction_arg,
        seed=CORRUPTION_SEED
    )


    result[
        "scenario_id"
    ] = scenario_id

    # Keep direction numerically explicit
    result[
        "direction"
    ] = direction_value


    results_records.append(
        result
    )


    # -----------------------------------
    # Save immediately after every case
    # -----------------------------------
    screening_results_df = pd.DataFrame(
        results_records
    )

    screening_results_df.to_csv(
        SCREENING_PATH,
        index=False
    )


    completed = len(
        screening_results_df
    )

    print(
        f"[{completed:3d}/{total_scenarios}] "
        f"{scenario_id} | "
        f"{corruption:7s} | "
        f"{sensor:9s} | "
        f"{severity:6s} | "
        f"dir={direction_value:+d} | "
        f"RMSE={result['full_RMSE']:.3f}"
    )


print(
    "\n315-scenario screening loop finished."
)

In [ ]:
screening_results_df = pd.read_csv(
    SCREENING_PATH
)

print(
    "Completed scenarios:",
    len(screening_results_df)
)

print(
    "Unique scenario IDs:",
    screening_results_df[
        "scenario_id"
    ].nunique()
)

print(
    "Missing RMSE values:",
    screening_results_df[
        "full_RMSE"
    ].isna().sum()
)

print(
    "Infinite RMSE values:",
    np.isinf(
        screening_results_df[
            "full_RMSE"
        ]
    ).sum()
)

In [ ]:
assert len(
    screening_results_df
) == 315

assert (
    screening_results_df[
        "scenario_id"
    ].nunique()
    == 315
)

assert (
    screening_results_df[
        "full_RMSE"
    ].isna().sum()
    == 0
)

assert (
    np.isfinite(
        screening_results_df[
            "full_RMSE"
        ]
    ).all()
)

print(
    "All 315 degradation-impact scenarios completed successfully."
)

In [ ]:
impact_summary = (
    screening_results_df
    .groupby("corruption")
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RMSE=(
            "full_RMSE",
            "mean"
        ),

        max_RMSE=(
            "full_RMSE",
            "max"
        ),

        mean_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        max_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        )
    )
    .sort_values(
        "max_RMSE",
        ascending=False
    )
)

display(
    impact_summary.round(3)
)

In [ ]:
worst_scenarios_df = (
    screening_results_df
    .sort_values(
        "full_RMSE_change_pct",
        ascending=False
    )
    [
        [
            "scenario_id",
            "corruption",
            "sensor",
            "severity",
            "level",
            "direction",
            "full_RMSE",
            "full_RMSE_delta",
            "full_RMSE_change_pct",
            "snapshot_RMSE",
            "snapshot_RMSE_change_pct"
        ]
    ]
    .head(20)
)

display(
    worst_scenarios_df.round(3)
)

In [ ]:
sensor_corruption_summary = (
    screening_results_df
    .groupby(
        [
            "sensor",
            "corruption"
        ]
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_full_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        max_full_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        ),

        mean_snapshot_RMSE_change_pct=(
            "snapshot_RMSE_change_pct",
            "mean"
        ),

        max_snapshot_RMSE_change_pct=(
            "snapshot_RMSE_change_pct",
            "max"
        )
    )
    .reset_index()
)

display(
    sensor_corruption_summary
    .sort_values(
        "max_full_RMSE_change_pct",
        ascending=False
    )
    .head(25)
    .round(3)
)

In [ ]:
sensor_vulnerability_df = (
    sensor_corruption_summary
    .groupby("sensor")
    .agg(
        mean_across_corruptions_pct=(
            "mean_full_RMSE_change_pct",
            "mean"
        ),

        worst_case_pct=(
            "max_full_RMSE_change_pct",
            "max"
        ),

        mean_snapshot_across_corruptions_pct=(
            "mean_snapshot_RMSE_change_pct",
            "mean"
        ),

        worst_snapshot_case_pct=(
            "max_snapshot_RMSE_change_pct",
            "max"
        )
    )
    .sort_values(
        "mean_across_corruptions_pct",
        ascending=False
    )
)

display(
    sensor_vulnerability_df.round(3)
)

In [ ]:
severity_summary_df = (
    screening_results_df
    .groupby(
        [
            "corruption",
            "severity"
        ]
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        median_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "median"
        ),

        max_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        )
    )
    .reset_index()
)

severity_order = {
    "low": 0,
    "medium": 1,
    "high": 2
}

severity_summary_df[
    "severity_order"
] = severity_summary_df[
    "severity"
].map(
    severity_order
)

severity_summary_df = (
    severity_summary_df
    .sort_values(
        [
            "corruption",
            "severity_order"
        ]
    )
    .drop(
        columns="severity_order"
    )
)

display(
    severity_summary_df.round(3)
)

In [ ]:
direction_summary_df = (
    screening_results_df[
        screening_results_df[
            "corruption"
        ].isin(
            [
                "bias",
                "drift"
            ]
        )
    ]
    .groupby(
        [
            "corruption",
            "direction"
        ]
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        median_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "median"
        ),

        max_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        )
    )
    .reset_index()
)

display(
    direction_summary_df.round(3)
)

In [ ]:
worst_scenarios_df.to_csv(
    STEP5_DIR
    / "fd001_top20_worst_screening_scenarios.csv",
    index=False
)

sensor_corruption_summary.to_csv(
    STEP5_DIR
    / "fd001_sensor_corruption_summary.csv",
    index=False
)

sensor_vulnerability_df.to_csv(
    STEP5_DIR
    / "fd001_sensor_vulnerability_ranking.csv"
)

severity_summary_df.to_csv(
    STEP5_DIR
    / "fd001_severity_summary.csv",
    index=False
)

direction_summary_df.to_csv(
    STEP5_DIR
    / "fd001_direction_summary.csv",
    index=False
)

print(
    "Step-5 screening-analysis tables saved."
)

In [ ]:
ROBUSTNESS_SEEDS = list(range(42, 52))

print("Robustness seeds:", ROBUSTNESS_SEEDS)
print("Number of seeds :", len(ROBUSTNESS_SEEDS))

stochastic_plan_df = (
    scenario_plan_df[
        scenario_plan_df["corruption"].isin(
            ["noise", "dropout"]
        )
    ]
    .copy()
)

print(
    "Base stochastic scenarios:",
    len(stochastic_plan_df)
)

print(
    "Total robustness evaluations:",
    len(stochastic_plan_df)
    * len(ROBUSTNESS_SEEDS)
)

assert len(stochastic_plan_df) == 90
assert len(ROBUSTNESS_SEEDS) == 10

print(
    "Stochastic robustness plan verified."
)

In [ ]:
robustness_plan_records = []

for _, row in stochastic_plan_df.iterrows():

    for seed in ROBUSTNESS_SEEDS:

        robustness_plan_records.append({
            "scenario_id":
                row["scenario_id"],

            "corruption":
                row["corruption"],

            "sensor":
                row["sensor"],

            "severity":
                row["severity"],

            "level":
                row["level"],

            "direction":
                int(row["direction"]),

            "seed":
                seed
        })


robustness_plan_df = pd.DataFrame(
    robustness_plan_records
)

assert len(robustness_plan_df) == 900

ROBUSTNESS_PLAN_PATH = (
    STEP5_DIR
    / "fd001_stochastic_robustness_plan.csv"
)

robustness_plan_df.to_csv(
    ROBUSTNESS_PLAN_PATH,
    index=False
)

print(
    "Robustness plan saved:"
)

print(
    ROBUSTNESS_PLAN_PATH
)

In [ ]:
ROBUSTNESS_RESULTS_PATH = (
    STEP5_DIR
    / "fd001_stochastic_robustness_results.csv"
)


if ROBUSTNESS_RESULTS_PATH.exists():

    robustness_results_df = pd.read_csv(
        ROBUSTNESS_RESULTS_PATH
    )

    completed_keys = set(
        zip(
            robustness_results_df[
                "scenario_id"
            ],

            robustness_results_df[
                "seed"
            ]
        )
    )

    robustness_records = (
        robustness_results_df
        .to_dict("records")
    )

    print(
        "Existing robustness runs:",
        len(completed_keys)
    )

else:

    completed_keys = set()
    robustness_records = []

    print(
        "Starting a new robustness run."
    )


total_runs = len(
    robustness_plan_df
)


for _, row in robustness_plan_df.iterrows():

    scenario_id = row[
        "scenario_id"
    ]

    seed = int(
        row["seed"]
    )

    key = (
        scenario_id,
        seed
    )

    if key in completed_keys:
        continue


    corruption = row[
        "corruption"
    ]

    sensor = row[
        "sensor"
    ]

    severity = row[
        "severity"
    ]


    (
        result,
        _,
        _
    ) = evaluate_corruption_scenario(
        corruption=corruption,
        sensor=sensor,
        severity_name=severity,
        direction=None,
        seed=seed
    )


    result[
        "scenario_id"
    ] = scenario_id

    result[
        "seed"
    ] = seed

    result[
        "direction"
    ] = 0


    robustness_records.append(
        result
    )


    robustness_results_df = (
        pd.DataFrame(
            robustness_records
        )
    )

    robustness_results_df.to_csv(
        ROBUSTNESS_RESULTS_PATH,
        index=False
    )


    completed = len(
        robustness_results_df
    )

    if (
        completed % 25 == 0
        or completed == total_runs
    ):
        print(
            f"[{completed:3d}/{total_runs}] "
            f"{corruption:7s} | "
            f"{sensor:9s} | "
            f"{severity:6s} | "
            f"seed={seed}"
        )


print(
    "\nStochastic robustness run finished."
)

In [ ]:
robustness_results_df = pd.read_csv(
    ROBUSTNESS_RESULTS_PATH
)

print(
    "Completed robustness runs:",
    len(robustness_results_df)
)

print(
    "Unique seeds:",
    sorted(
        robustness_results_df[
            "seed"
        ].unique()
    )
)

print(
    "Missing RMSE:",
    robustness_results_df[
        "full_RMSE"
    ].isna().sum()
)

assert len(
    robustness_results_df
) == 900

assert (
    robustness_results_df[
        ["scenario_id", "seed"]
    ]
    .drop_duplicates()
    .shape[0]
    == 900
)

assert np.isfinite(
    robustness_results_df[
        "full_RMSE"
    ]
).all()

print(
    "All 900 stochastic robustness runs verified."
)

In [ ]:
stochastic_robustness_summary = (
    robustness_results_df
    .groupby(
        [
            "corruption",
            "sensor",
            "severity"
        ]
    )
    .agg(
        runs=(
            "seed",
            "count"
        ),

        mean_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        std_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "std"
        ),

        min_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "min"
        ),

        max_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        )
    )
    .reset_index()
)

display(
    stochastic_robustness_summary
    .sort_values(
        "mean_RMSE_change_pct",
        ascending=False
    )
    .head(25)
    .round(3)
)

In [ ]:
robustness_by_corruption = (
    robustness_results_df
    .groupby("corruption")
    .agg(
        evaluations=(
            "scenario_id",
            "count"
        ),

        mean_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        std_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "std"
        ),

        max_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "max"
        )
    )
)

display(
    robustness_by_corruption.round(3)
)

In [ ]:
stochastic_robustness_summary.to_csv(
    STEP5_DIR
    / "fd001_stochastic_robustness_summary.csv",
    index=False
)

robustness_by_corruption.to_csv(
    STEP5_DIR
    / "fd001_stochastic_robustness_by_corruption.csv"
)

print(
    "Stochastic robustness summaries saved."
)

In [ ]:
def calculate_safety_metrics(
    true_rul,
    predictions,
    eol_threshold=30,
    dangerous_margin=10
):
    true_rul = np.asarray(
        true_rul,
        dtype=float
    )

    predictions = np.asarray(
        predictions,
        dtype=float
    )

    assert len(true_rul) == len(predictions)

    # ---------------------------------
    # Near-end-of-life samples
    # ---------------------------------
    near_eol_mask = (
        true_rul <= eol_threshold
    )

    y_true = true_rul[
        near_eol_mask
    ]

    y_pred = predictions[
        near_eol_mask
    ]

    # Positive error = RUL overestimate
    signed_error = (
        y_pred - y_true
    )

    dangerous_mask = (
        signed_error
        > dangerous_margin
    )

    rmse = np.sqrt(
        np.mean(
            signed_error ** 2
        )
    )

    mae = np.mean(
        np.abs(
            signed_error
        )
    )

    return {
        "near_eol_samples":
            len(y_true),

        "near_eol_RMSE":
            rmse,

        "near_eol_MAE":
            mae,

        "near_eol_mean_error":
            signed_error.mean(),

        "over10_count":
            dangerous_mask.sum(),

        "over10_rate_pct":
            100
            * dangerous_mask.mean(),

        "max_overestimation":
            max(
                0.0,
                signed_error.max()
            )
    }


print(
    "Safety metric function ready."
)

In [ ]:
print(
    "RUL_raw available:",
    "RUL_raw" in val_seq_meta.columns
)

print(
    "Metadata rows:",
    len(val_seq_meta)
)

assert (
    "RUL_raw"
    in val_seq_meta.columns
)

assert (
    len(val_seq_meta)
    == len(clean_predictions)
)

In [ ]:
clean_safety_metrics = (
    calculate_safety_metrics(
        true_rul=
            val_seq_meta[
                "RUL_raw"
            ].to_numpy(),

        predictions=
            clean_predictions,

        eol_threshold=30,

        dangerous_margin=10
    )
)


print(
    "Clean near-EOL safety metrics:"
)

for metric, value in (
    clean_safety_metrics.items()
):
    print(
        f"{metric}: {value}"
    )

In [ ]:
assert (
    clean_safety_metrics[
        "near_eol_samples"
    ]
    == 620
)

assert (
    clean_safety_metrics[
        "over10_count"
    ]
    == 56
)

assert np.isclose(
    clean_safety_metrics[
        "near_eol_mean_error"
    ],
    2.401,
    atol=0.01
)

print(
    "Clean near-EOL safety baseline reproduced."
)

In [ ]:
SAFETY_RESULTS_PATH = (
    STEP5_DIR
    / "fd001_degradation_safety_screening_seed42.csv"
)


if SAFETY_RESULTS_PATH.exists():

    safety_results_df = pd.read_csv(
        SAFETY_RESULTS_PATH
    )

    completed_ids = set(
        safety_results_df[
            "scenario_id"
        ]
    )

    safety_records = (
        safety_results_df
        .to_dict("records")
    )

    print(
        "Existing safety results:",
        len(completed_ids)
    )

else:

    completed_ids = set()
    safety_records = []

    print(
        "Starting a new safety screening."
    )


total_scenarios = len(
    scenario_plan_df
)


for _, row in scenario_plan_df.iterrows():

    scenario_id = row[
        "scenario_id"
    ]

    if scenario_id in completed_ids:
        continue


    direction_value = int(
        row["direction"]
    )

    direction_arg = (
        None
        if direction_value == 0
        else direction_value
    )


    (
        degradation_result,
        predictions,
        _
    ) = evaluate_corruption_scenario(
        corruption=
            row["corruption"],

        sensor=
            row["sensor"],

        severity_name=
            row["severity"],

        direction=
            direction_arg,

        seed=
            CORRUPTION_SEED
    )


    safety = calculate_safety_metrics(
        true_rul=
            val_seq_meta[
                "RUL_raw"
            ].to_numpy(),

        predictions=
            predictions,

        eol_threshold=30,

        dangerous_margin=10
    )


    record = {
        "scenario_id":
            scenario_id,

        "corruption":
            row["corruption"],

        "sensor":
            row["sensor"],

        "severity":
            row["severity"],

        "level":
            row["level"],

        "direction":
            direction_value,

        "full_RMSE_change_pct":
            degradation_result[
                "full_RMSE_change_pct"
            ],

        **safety
    }


    record[
        "over10_rate_delta_pp"
    ] = (
        safety[
            "over10_rate_pct"
        ]
        - clean_safety_metrics[
            "over10_rate_pct"
        ]
    )


    record[
        "mean_error_delta"
    ] = (
        safety[
            "near_eol_mean_error"
        ]
        - clean_safety_metrics[
            "near_eol_mean_error"
        ]
    )


    safety_records.append(
        record
    )


    safety_results_df = pd.DataFrame(
        safety_records
    )

    safety_results_df.to_csv(
        SAFETY_RESULTS_PATH,
        index=False
    )


    completed = len(
        safety_results_df
    )

    if (
        completed % 25 == 0
        or completed == total_scenarios
    ):

        print(
            f"[{completed:3d}/{total_scenarios}] "
            f"{scenario_id} | "
            f"{row['corruption']:7s} | "
            f"{row['sensor']:9s} | "
            f"{row['severity']:6s}"
        )


print(
    "\n315-scenario safety screening finished."
)

In [ ]:
safety_results_df = pd.read_csv(
    SAFETY_RESULTS_PATH
)

print(
    "Safety scenarios:",
    len(safety_results_df)
)

print(
    "Unique scenario IDs:",
    safety_results_df[
        "scenario_id"
    ].nunique()
)

print(
    "Missing safety values:",
    safety_results_df.isna()
    .sum()
    .sum()
)

assert len(
    safety_results_df
) == 315

assert (
    safety_results_df[
        "scenario_id"
    ].nunique()
    == 315
)

print(
    "All 315 safety scenarios verified."
)

In [ ]:
dangerous_scenarios_df = (
    safety_results_df
    .sort_values(
        [
            "over10_rate_delta_pp",
            "max_overestimation"
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(20)
)


display(
    dangerous_scenarios_df[
        [
            "scenario_id",
            "corruption",
            "sensor",
            "severity",
            "direction",
            "full_RMSE_change_pct",
            "near_eol_RMSE",
            "near_eol_mean_error",
            "over10_count",
            "over10_rate_pct",
            "over10_rate_delta_pp",
            "max_overestimation"
        ]
    ].round(3)
)

In [ ]:
dangerous_compact_df = (
    safety_results_df
    .sort_values(
        [
            "over10_rate_delta_pp",
            "max_overestimation"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "scenario_id",
            "corruption",
            "sensor",
            "severity",
            "direction",
            "full_RMSE_change_pct",
            "near_eol_mean_error",
            "over10_rate_pct",
            "over10_rate_delta_pp",
            "max_overestimation"
        ]
    ]
    .head(20)
)

display(
    dangerous_compact_df.round(3)
)

In [ ]:
safety_by_corruption_df = (
    safety_results_df
    .groupby("corruption")
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_full_RMSE_change_pct=(
            "full_RMSE_change_pct",
            "mean"
        ),

        mean_over10_rate_pct=(
            "over10_rate_pct",
            "mean"
        ),

        mean_over10_delta_pp=(
            "over10_rate_delta_pp",
            "mean"
        ),

        max_over10_delta_pp=(
            "over10_rate_delta_pp",
            "max"
        ),

        mean_max_overestimation=(
            "max_overestimation",
            "mean"
        ),

        worst_max_overestimation=(
            "max_overestimation",
            "max"
        )
    )
    .sort_values(
        "max_over10_delta_pp",
        ascending=False
    )
)

display(
    safety_by_corruption_df.round(3)
)

In [ ]:
sensor_safety_df = (
    safety_results_df
    .groupby("sensor")
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_over10_delta_pp=(
            "over10_rate_delta_pp",
            "mean"
        ),

        max_over10_delta_pp=(
            "over10_rate_delta_pp",
            "max"
        ),

        mean_near_eol_error_delta=(
            "mean_error_delta",
            "mean"
        ),

        max_near_eol_error_delta=(
            "mean_error_delta",
            "max"
        ),

        worst_max_overestimation=(
            "max_overestimation",
            "max"
        )
    )
    .sort_values(
        "max_over10_delta_pp",
        ascending=False
    )
)

display(
    sensor_safety_df.round(3)
)

In [ ]:
pearson_corr = (
    safety_results_df[
        [
            "full_RMSE_change_pct",
            "over10_rate_delta_pp"
        ]
    ]
    .corr(
        method="pearson"
    )
    .iloc[0, 1]
)

spearman_corr = (
    safety_results_df[
        [
            "full_RMSE_change_pct",
            "over10_rate_delta_pp"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[0, 1]
)


print(
    "Pearson correlation:",
    round(
        pearson_corr,
        3
    )
)

print(
    "Spearman correlation:",
    round(
        spearman_corr,
        3
    )
)

In [ ]:
hidden_risk_df = (
    safety_results_df[
        (
            safety_results_df[
                "full_RMSE_change_pct"
            ] < 1.0
        )
        &
        (
            safety_results_df[
                "over10_rate_delta_pp"
            ] > 0
        )
    ]
    .sort_values(
        "over10_rate_delta_pp",
        ascending=False
    )
    [
        [
            "scenario_id",
            "corruption",
            "sensor",
            "severity",
            "direction",
            "full_RMSE_change_pct",
            "near_eol_mean_error",
            "over10_rate_pct",
            "over10_rate_delta_pp",
            "max_overestimation"
        ]
    ]
)

print(
    "Hidden-risk scenarios:",
    len(hidden_risk_df)
)

display(
    hidden_risk_df
    .head(20)
    .round(3)
)

In [ ]:
dangerous_compact_df.to_csv(
    STEP5_DIR
    / "fd001_top20_dangerous_near_eol_scenarios.csv",
    index=False
)

safety_by_corruption_df.to_csv(
    STEP5_DIR
    / "fd001_safety_by_corruption.csv"
)

sensor_safety_df.to_csv(
    STEP5_DIR
    / "fd001_sensor_safety_vulnerability.csv"
)

hidden_risk_df.to_csv(
    STEP5_DIR
    / "fd001_hidden_risk_scenarios.csv",
    index=False
)

print(
    "Step-5 safety-analysis tables saved."
)

In [ ]:
print("Total hidden-risk scenarios:", len(hidden_risk_df))

print("\nHidden-risk scenarios by corruption:")
display(
    hidden_risk_df[
        "corruption"
    ]
    .value_counts()
    .rename("scenarios")
    .to_frame()
)